# Tables from PDFs (Colab)

**CEEW Scraping Field Course, Track B, lesson B2.**

Many Indian agencies publish data only as PDF bulletins. This notebook pulls the tables out of a PDF and saves them to a Google Sheet, so you can check and use them.

It works on PDFs that contain real text. It can't read scanned images: Step 3 tells you which pages look scanned. For those, ask Claude to read the page image instead, then check every number.

**Practise first** on [`practice_bulletin_safdarjung.pdf`](https://mrafiuddin50.github.io/ceew_scraping_curriculum/practice/practice_bulletin_safdarjung.pdf): download it and upload it to your Google Drive. The answers are on the [course page](https://mrafiuddin50.github.io/ceew_scraping_curriculum/#trackb) under lesson B2.

**Status:** tested on 12 October 2026 against the practice PDF. Real bulletins vary; always compare a few numbers with the PDF by eye.

In [ ]:
#@title Step 1: Settings { display-mode: "form" }
SOURCE = "Google Drive file" #@param ["Google Drive file", "Web link"]
#@markdown For a Drive file, the path inside your Drive, e.g. `MyDrive/course/practice_bulletin_safdarjung.pdf`
DRIVE_PATH = "MyDrive/practice_bulletin_safdarjung.pdf" #@param {type:"string"}
#@markdown For a web link, the direct link to the PDF.
PDF_URL = "" #@param {type:"string"}
#@markdown Pages to read: `all`, or e.g. `1-3, 5`
PAGES = "all" #@param {type:"string"}
print("Settings saved. Run Step 2.")

In [ ]:
#@title Step 2: Install the PDF reader and load the working parts { display-mode: "form" }
!pip -q install pdfplumber

import io, re, time, datetime as dt
import pdfplumber, pandas as pd, requests

def load_pdf_bytes(source, drive_path, url):
    if source == "Google Drive file":
        from google.colab import drive
        drive.mount("/content/drive")
        import os
        rel = drive_path.strip().lstrip("/").replace("My Drive/", "MyDrive/", 1)
        if not rel.startswith(("MyDrive/", "Shareddrives/")):
            rel = "MyDrive/" + rel
        path = "/content/drive/" + rel
        if not os.path.exists(path):
            top = sorted(os.listdir("/content/drive/MyDrive"))[:40]
            raise FileNotFoundError(f"No file at {rel}. Top of your Drive: {', '.join(top)}")
        with open(path, "rb") as f:
            return f.read(), path
    r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0 (compatible; CEEW-research-collector/2.0)"})
    r.raise_for_status()
    if not r.content.startswith(b"%PDF"):
        raise ValueError("That link did not return a PDF. Open it in your browser and copy the direct PDF link.")
    return r.content, url

def page_numbers(spec, n):
    if spec.strip().lower() in ("", "all"):
        return list(range(1, n + 1))
    out = []
    for part in spec.split(","):
        a, _, b = part.strip().partition("-")
        out += list(range(int(a), int(b or a) + 1))
    return [p for p in out if 1 <= p <= n]

def inspect(pdf):
    """Pages with almost no text are probably scanned images; pdfplumber can't read those."""
    report = []
    for i, page in enumerate(pdf.pages, 1):
        chars = len((page.extract_text() or "").strip())
        report.append({"page": i, "characters_of_text": chars, "looks_scanned": chars < 20})
    return pd.DataFrame(report)

def tables_from(pdf, pages):
    found = []
    for p in pages:
        for k, t in enumerate(pdf.pages[p - 1].extract_tables(), 1):
            if not t or len(t) < 2:
                continue
            header = [(h or f"col{j+1}").replace("\n", " ").strip() for j, h in enumerate(t[0])]
            body = [[(c or "").replace("\n", " ").strip() for c in row] for row in t[1:]]
            found.append({"page": p, "table": k, "df": pd.DataFrame(body, columns=header)})
    return found

print('Ready. Run Step 3.')

In [ ]:
#@title Step 3: Open the PDF and check it has text { display-mode: "form" }
data, where = load_pdf_bytes(SOURCE, DRIVE_PATH, PDF_URL)
pdf = pdfplumber.open(io.BytesIO(data))
report = inspect(pdf)
print(f"Opened {where}: {len(pdf.pages)} page(s).")
print(report.to_string(index=False))
if report["looks_scanned"].any():
    print("\nSome pages look scanned. Tables on those pages won't be found; ask Claude to read those page images instead.")

In [ ]:
#@title Step 4: Find the tables { display-mode: "form" }
found = tables_from(pdf, page_numbers(PAGES, len(pdf.pages)))
print(f"Found {len(found)} table(s).")
for t in found:
    print(f"\n--- Page {t['page']}, table {t['table']}: {len(t['df'])} rows x {len(t['df'].columns)} columns")
    print(t["df"].head(5).to_string(index=False))
print("\nCompare a few numbers above with the PDF itself before saving.")

In [ ]:
#@title Step 5: Save to a Google Sheet { display-mode: "form" }
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
stamp = dt.datetime.now(dt.timezone(dt.timedelta(hours=5, minutes=30))).strftime("%Y-%m-%d %H%M IST")
sh = gc.create(f"PDF tables {stamp}")
about = [["source", where], ["extracted (IST)", stamp], ["pages read", PAGES], ["tables found", str(len(found))],
         ["caveat", "Extracted automatically. Check totals and a sample of values against the PDF before use."]]
sh.sheet1.update_title("about")
sh.sheet1.update(values=about, range_name="A1")
for t in found:
    df = t["df"]
    values = [list(df.columns)] + df.values.tolist()
    ws = sh.add_worksheet(title=f"p{t['page']}_t{t['table']}", rows=len(values) + 5, cols=len(df.columns) + 2)
    ws.update(values=values, range_name="A1")
print(f"Google Sheet created: {sh.url}")

## Check what came out

PDF tables break in predictable ways. Look for:
- **Header rows** split across two rows, or repeated at the top of each page.
- **Merged cells** that leave blanks in rows below them.
- **Numbers as text**: commas (`1,234`), units in the cell (`45 MW`), or footnote marks (`12.5*`).
- **Page footers** or notes picked up as table rows.
- **Totals rows** that shouldn't be added up with the rest.

Then compare: does the table's own total (if it has one) match the sum of its rows? Record what you checked in your scraping log.